# Onboarding Intake, Document Completeness, and Next-Step Guidance

**Objective** - Build a realistic onboarding workflow that:

1. Collects onboarding details from applicants or new hires
2. Validates required fields and checks uploaded-document completeness
3. Flags missing, expired, or low-quality submissions
4. Generates personalised next-step guidance for operations teams and applicants
5. Produces analytics that help onboarding teams prioritise follow-up

This notebook demonstrates an end-to-end operational workflow using synthetic but realistic data.

## Why This Workflow Matters

Onboarding workflows often fail in the handoff between intake, verification, and follow-up.
Common issues include:

| Problem | Operational impact |
|---|---|
| Missing profile fields | Delays verification and account activation |
| Incomplete document packets | Manual back-and-forth with the applicant |
| Expired IDs or outdated forms | Compliance and fraud risk |
| No prioritisation | Teams chase easy cases instead of blocked ones |
| Generic follow-up emails | Low completion rates and poor applicant experience |

A good onboarding workflow turns raw submissions into structured case status, clear next actions, and measurable throughput.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import math
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print('Libraries loaded.')

## 1. Intake Schema

We model each onboarding case with three layers:

- **Applicant profile**: identity, region, onboarding type, role, start date
- **Submitted documents**: IDs, forms, certifications, agreements
- **Workflow state**: verification risk, completeness score, recommended next actions

The example is framed as a mixed workforce onboarding pipeline for employees and contractors.

In [ ]:
@dataclass
class SubmittedDocument:
    name: str
    status: str
    uploaded_at: str | None = None
    expires_on: str | None = None
    quality_score: float | None = None
    notes: str = ''


@dataclass
class OnboardingCase:
    case_id: str
    onboarding_type: str
    full_name: str
    role_family: str
    region: str
    start_date: str
    manager: str
    email: str
    phone: str
    citizenship: str
    work_location: str
    documents: list[SubmittedDocument] = field(default_factory=list)

    def profile_text(self) -> str:
        return (
            f'{self.onboarding_type} | {self.role_family} | {self.region} | '
            f'{self.work_location} | {self.citizenship}'
        )

## 2. Synthetic Intake Cases

We create 20 onboarding cases with varied completeness patterns.
Some are clean and ready for activation; others are blocked by missing tax forms, expired IDs, or unreadable uploads.

In [ ]:
BASE_DATE = datetime(2026, 4, 12)

def iso_date(days_offset: int) -> str:
    return (BASE_DATE + timedelta(days=days_offset)).date().isoformat()


CASE_DEFS = [
    {
        'case_id': 'ONB-1001', 'onboarding_type': 'employee', 'full_name': 'Ava Patel',
        'role_family': 'engineering', 'region': 'US', 'start_date': iso_date(10),
        'manager': 'Jordan Kim', 'email': 'ava.patel@example.com', 'phone': '+1-555-0101',
        'citizenship': 'US', 'work_location': 'remote',
        'documents': [
            ('government_id', 'accepted', -5, 400, 0.98, ''),
            ('signed_offer', 'accepted', -7, None, 0.99, ''),
            ('tax_form', 'accepted', -4, None, 0.97, ''),
            ('nda', 'accepted', -6, None, 0.99, ''),
            ('background_consent', 'accepted', -6, None, 0.96, ''),
        ],
    },
    {
        'case_id': 'ONB-1002', 'onboarding_type': 'contractor', 'full_name': 'Mateo Ruiz',
        'role_family': 'design', 'region': 'UK', 'start_date': iso_date(5),
        'manager': 'Iris Bell', 'email': 'mateo.ruiz@example.com', 'phone': '+44-20-555-0102',
        'citizenship': 'ES', 'work_location': 'london',
        'documents': [
            ('government_id', 'accepted', -3, 20, 0.91, ''),
            ('msa', 'accepted', -8, None, 0.95, ''),
            ('bank_details', 'missing', None, None, None, 'Awaiting upload'),
            ('nda', 'accepted', -7, None, 0.97, ''),
        ],
    },
    {
        'case_id': 'ONB-1003', 'onboarding_type': 'employee', 'full_name': 'Noah Brooks',
        'role_family': 'finance', 'region': 'US', 'start_date': iso_date(3),
        'manager': 'Renee Fox', 'email': 'noah.brooks@example.com', 'phone': '+1-555-0103',
        'citizenship': 'US', 'work_location': 'new_york',
        'documents': [
            ('government_id', 'uploaded', -2, -1, 0.92, 'ID expires before start date'),
            ('signed_offer', 'accepted', -8, None, 0.99, ''),
            ('tax_form', 'accepted', -2, None, 0.91, ''),
            ('nda', 'accepted', -6, None, 0.98, ''),
            ('background_consent', 'accepted', -5, None, 0.96, ''),
        ],
    },
    {
        'case_id': 'ONB-1004', 'onboarding_type': 'employee', 'full_name': 'Lina Hassan',
        'role_family': 'support', 'region': 'UAE', 'start_date': iso_date(14),
        'manager': 'Mia Chen', 'email': 'lina.hassan@example.com', 'phone': '+971-4-555-0104',
        'citizenship': 'EG', 'work_location': 'dubai',
        'documents': [
            ('government_id', 'accepted', -3, 700, 0.88, ''),
            ('signed_offer', 'accepted', -9, None, 0.97, ''),
            ('tax_form', 'missing', None, None, None, 'Missing labor registration form'),
            ('nda', 'accepted', -7, None, 0.96, ''),
            ('background_consent', 'review', -2, None, 0.82, 'Signature mismatch'),
        ],
    },
    {
        'case_id': 'ONB-1005', 'onboarding_type': 'contractor', 'full_name': 'Sara Jensen',
        'role_family': 'marketing', 'region': 'DE', 'start_date': iso_date(7),
        'manager': 'Elena Cruz', 'email': 'sara.jensen@example.com', 'phone': '+49-30-555-0105',
        'citizenship': 'DE', 'work_location': 'berlin',
        'documents': [
            ('government_id', 'accepted', -5, 365, 0.96, ''),
            ('msa', 'accepted', -8, None, 0.95, ''),
            ('bank_details', 'accepted', -6, None, 0.92, ''),
            ('nda', 'accepted', -7, None, 0.97, ''),
        ],
    },
    {
        'case_id': 'ONB-1006', 'onboarding_type': 'employee', 'full_name': 'Ethan Park',
        'role_family': 'engineering', 'region': 'CA', 'start_date': iso_date(1),
        'manager': 'Jordan Kim', 'email': 'ethan.park@example.com', 'phone': '+1-416-555-0106',
        'citizenship': 'CA', 'work_location': 'toronto',
        'documents': [
            ('government_id', 'accepted', -4, 180, 0.95, ''),
            ('signed_offer', 'accepted', -10, None, 0.98, ''),
            ('tax_form', 'review', -3, None, 0.71, 'Address partially cut off'),
            ('nda', 'accepted', -8, None, 0.98, ''),
            ('background_consent', 'accepted', -8, None, 0.97, ''),
        ],
    },
    {
        'case_id': 'ONB-1007', 'onboarding_type': 'employee', 'full_name': 'Priya Menon',
        'role_family': 'operations', 'region': 'IN', 'start_date': iso_date(9),
        'manager': 'Dylan Hart', 'email': 'priya.menon@example.com', 'phone': '+91-80-555-0107',
        'citizenship': 'IN', 'work_location': 'bangalore',
        'documents': [
            ('government_id', 'accepted', -5, 540, 0.93, ''),
            ('signed_offer', 'accepted', -12, None, 0.97, ''),
            ('tax_form', 'accepted', -7, None, 0.94, ''),
            ('nda', 'accepted', -11, None, 0.98, ''),
            ('background_consent', 'accepted', -10, None, 0.96, ''),
            ('education_verification', 'missing', None, None, None, 'University proof pending'),
        ],
    },
    {
        'case_id': 'ONB-1008', 'onboarding_type': 'contractor', 'full_name': 'Julia Novak',
        'role_family': 'analytics', 'region': 'PL', 'start_date': iso_date(12),
        'manager': 'Renee Fox', 'email': 'julia.novak@example.com', 'phone': '+48-22-555-0108',
        'citizenship': 'PL', 'work_location': 'warsaw',
        'documents': [
            ('government_id', 'review', -1, 100, 0.63, 'Image blurred'),
            ('msa', 'accepted', -9, None, 0.96, ''),
            ('bank_details', 'accepted', -8, None, 0.89, ''),
            ('nda', 'accepted', -9, None, 0.97, ''),
        ],
    },
    {
        'case_id': 'ONB-1009', 'onboarding_type': 'employee', 'full_name': 'Omar Farouk',
        'role_family': 'sales', 'region': 'AE', 'start_date': iso_date(4),
        'manager': 'Iris Bell', 'email': 'omar.farouk@example.com', 'phone': '+971-4-555-0109',
        'citizenship': 'AE', 'work_location': 'abu_dhabi',
        'documents': [
            ('government_id', 'accepted', -2, 300, 0.95, ''),
            ('signed_offer', 'accepted', -6, None, 0.98, ''),
            ('tax_form', 'accepted', -5, None, 0.94, ''),
            ('nda', 'accepted', -6, None, 0.96, ''),
            ('background_consent', 'accepted', -4, None, 0.95, ''),
        ],
    },
    {
        'case_id': 'ONB-1010', 'onboarding_type': 'employee', 'full_name': 'Grace Liu',
        'role_family': 'engineering', 'region': 'US', 'start_date': iso_date(2),
        'manager': 'Jordan Kim', 'email': 'grace.liu@example.com', 'phone': '+1-555-0110',
        'citizenship': 'US', 'work_location': 'seattle',
        'documents': [
            ('government_id', 'accepted', -4, 600, 0.98, ''),
            ('signed_offer', 'accepted', -7, None, 0.99, ''),
            ('tax_form', 'accepted', -4, None, 0.95, ''),
            ('nda', 'accepted', -6, None, 0.99, ''),
            ('background_consent', 'accepted', -5, None, 0.97, ''),
            ('equipment_acknowledgement', 'missing', None, None, None, 'Laptop policy not signed'),
        ],
    },
]

# Extend to 20 cases by cloning patterns with small changes
extra_defs = []
for idx, seed in enumerate(CASE_DEFS[:10], start=11):
    clone = dict(seed)
    clone['case_id'] = f'ONB-{1000 + idx}'
    clone['full_name'] = seed['full_name'] + ' Jr'
    clone['email'] = seed['email'].replace('@', f'.{idx}@')
    clone['start_date'] = iso_date((idx % 9) + 1)
    docs = []
    for name, status, uploaded_offset, expiry_offset, quality, notes in seed['documents']:
        new_status = status
        new_notes = notes
        if idx % 3 == 0 and name == 'government_id':
            new_status = 'review'
            quality = min(0.79, quality if quality is not None else 0.76)
            new_notes = 'Needs manual identity review'
        if idx % 4 == 0 and 'tax_form' in name:
            new_status = 'missing'
            quality = None
            new_notes = 'Awaiting jurisdiction-specific tax form'
        docs.append((name, new_status, uploaded_offset, expiry_offset, quality, new_notes))
    clone['documents'] = docs
    extra_defs.append(clone)

CASE_DEFS.extend(extra_defs)

cases = []
for raw in CASE_DEFS:
    docs = []
    for name, status, uploaded_offset, expiry_offset, quality, notes in raw['documents']:
        uploaded_at = None if uploaded_offset is None else iso_date(uploaded_offset)
        expires_on = None if expiry_offset is None else iso_date(expiry_offset)
        docs.append(SubmittedDocument(name, status, uploaded_at, expires_on, quality, notes))
    payload = dict(raw)
    payload['documents'] = docs
    cases.append(OnboardingCase(**payload))

print(f'Cases created: {len(cases)}')
print(Counter(case.onboarding_type for case in cases))
print(Counter(case.role_family for case in cases))

## 3. Requirements Matrix

Different onboarding types require different document bundles.

- Employees need hiring and compliance forms
- Contractors need contract and payout details
- Certain roles need additional verification or policy acknowledgements

In [ ]:
BASE_REQUIREMENTS = {
    'employee': ['government_id', 'signed_offer', 'tax_form', 'nda', 'background_consent'],
    'contractor': ['government_id', 'msa', 'bank_details', 'nda'],
}

ROLE_REQUIREMENTS = {
    'engineering': ['equipment_acknowledgement'],
    'finance': ['conflict_of_interest_form'],
    'operations': ['education_verification'],
    'analytics': ['data_access_policy'],
}

REGION_REQUIREMENTS = {
    'US': ['i9_equivalent'],
    'UK': ['right_to_work_check'],
    'DE': ['tax_id_confirmation'],
}

def required_documents(case: OnboardingCase) -> list[str]:
    required = list(BASE_REQUIREMENTS[case.onboarding_type])
    required.extend(ROLE_REQUIREMENTS.get(case.role_family, []))
    required.extend(REGION_REQUIREMENTS.get(case.region, []))
    return sorted(set(required))

for case in cases[:3]:
    print(case.case_id, required_documents(case))

## 4. Document Completeness Engine

The completeness engine evaluates each case on four dimensions:

1. **Presence** - required documents uploaded or not
2. **Acceptance** - accepted vs still in review
3. **Freshness** - expired or expiring before the start date
4. **Quality** - low-confidence scans or partial forms

This produces a completeness score and a workflow status.

In [ ]:
@dataclass
class CaseAssessment:
    case: OnboardingCase
    required_docs: list[str]
    missing_docs: list[str]
    review_docs: list[str]
    expired_docs: list[str]
    low_quality_docs: list[str]
    completeness_score: float
    status: str
    risk_level: str
    next_steps: list[str] = field(default_factory=list)


def assess_case(case: OnboardingCase) -> CaseAssessment:
    req = required_documents(case)
    by_name = {doc.name: doc for doc in case.documents}

    missing_docs = [name for name in req if name not in by_name or by_name[name].status == 'missing']
    review_docs = []
    expired_docs = []
    low_quality_docs = []
    start_date = datetime.fromisoformat(case.start_date).date()

    for name in req:
        doc = by_name.get(name)
        if not doc or doc.status == 'missing':
            continue
        if doc.status == 'review' or doc.status == 'uploaded':
            review_docs.append(name)
        if doc.expires_on:
            expiry = datetime.fromisoformat(doc.expires_on).date()
            if expiry < start_date:
                expired_docs.append(name)
        if doc.quality_score is not None and doc.quality_score < 0.80:
            low_quality_docs.append(name)

    total_required = max(len(req), 1)
    penalties = (
        len(missing_docs) * 0.35
        + len(review_docs) * 0.15
        + len(expired_docs) * 0.20
        + len(low_quality_docs) * 0.10
    )
    completeness_score = max(0.0, round(1.0 - penalties / total_required, 3))

    if missing_docs or expired_docs:
        status = 'blocked'
    elif review_docs or low_quality_docs:
        status = 'review_needed'
    else:
        status = 'ready'

    if completeness_score < 0.45:
        risk_level = 'high'
    elif completeness_score < 0.75:
        risk_level = 'medium'
    else:
        risk_level = 'low'

    return CaseAssessment(
        case=case, required_docs=req, missing_docs=missing_docs, review_docs=review_docs,
        expired_docs=expired_docs, low_quality_docs=low_quality_docs,
        completeness_score=completeness_score, status=status, risk_level=risk_level,
    )

assessments = [assess_case(case) for case in cases]
print(Counter(a.status for a in assessments))
print(Counter(a.risk_level for a in assessments))

## 5. Guidance Generator

Operational value comes from turning assessment outputs into clear next steps.
The guidance generator creates two streams:

- **Applicant guidance**: what the person must upload or correct
- **Operations guidance**: what the onboarding team should verify, escalate, or approve

The goal is specific, case-aware guidance instead of generic reminders.

In [ ]:
FRIENDLY_NAMES = {
    'government_id': 'government-issued ID',
    'signed_offer': 'signed offer letter',
    'tax_form': 'tax form',
    'nda': 'NDA',
    'background_consent': 'background-check consent form',
    'msa': 'master services agreement',
    'bank_details': 'bank details form',
    'equipment_acknowledgement': 'equipment acknowledgement',
    'education_verification': 'education verification document',
    'data_access_policy': 'data access policy acknowledgement',
    'conflict_of_interest_form': 'conflict of interest form',
    'i9_equivalent': 'employment eligibility form',
    'right_to_work_check': 'right-to-work check',
    'tax_id_confirmation': 'tax ID confirmation',
}

def human_name(doc_name: str) -> str:
    return FRIENDLY_NAMES.get(doc_name, doc_name.replace('_', ' '))


def generate_next_steps(assessment: CaseAssessment) -> CaseAssessment:
    steps = []

    if assessment.status == 'ready':
        steps.append('Mark case ready for activation and queue account provisioning.')
        steps.append('Send welcome package and first-day instructions.')
    else:
        if assessment.missing_docs:
            missing_text = ', '.join(human_name(d) for d in assessment.missing_docs)
            steps.append(f'Request missing documents: {missing_text}.')
        if assessment.expired_docs:
            expired_text = ', '.join(human_name(d) for d in assessment.expired_docs)
            steps.append(f'Request renewed replacement for expired items: {expired_text}.')
        if assessment.review_docs:
            review_text = ', '.join(human_name(d) for d in assessment.review_docs)
            steps.append(f'Queue manual verification for: {review_text}.')
        if assessment.low_quality_docs:
            quality_text = ', '.join(human_name(d) for d in assessment.low_quality_docs)
            steps.append(f'Ask applicant to re-upload clearer copies of: {quality_text}.')

    days_to_start = (datetime.fromisoformat(assessment.case.start_date).date() - BASE_DATE.date()).days
    if days_to_start <= 3 and assessment.status != 'ready':
        steps.append('Escalate to onboarding operations lead due to upcoming start date.')
    elif assessment.risk_level == 'high':
        steps.append('Assign priority follow-up within 4 business hours.')
    else:
        steps.append('Send automated reminder and re-check status in 24 hours.')

    assessment.next_steps = steps
    return assessment

assessments = [generate_next_steps(a) for a in assessments]

sample = assessments[1]
print(sample.case.case_id, sample.status, sample.completeness_score)
for step in sample.next_steps:
    print('-', step)

## 6. End-to-End Workflow View

The full workflow looks like this:

```
Applicant submits details
        |
        v
Profile validation -> Required-document lookup -> Document assessment
        |                                      |
        +----------------------+---------------+
                               v
                     Case status classification
                               |
                               v
                 Next-step guidance generation
                               |
                  +------------+-------------+
                  |                          |
                  v                          v
        Applicant reminder            Ops work queue / escalation
```

In [ ]:
rows = []
for assessment in assessments:
    rows.append({
        'case_id': assessment.case.case_id,
        'name': assessment.case.full_name,
        'type': assessment.case.onboarding_type,
        'role_family': assessment.case.role_family,
        'region': assessment.case.region,
        'start_date': assessment.case.start_date,
        'status': assessment.status,
        'risk_level': assessment.risk_level,
        'completeness_score': assessment.completeness_score,
        'missing_count': len(assessment.missing_docs),
        'review_count': len(assessment.review_docs),
        'expired_count': len(assessment.expired_docs),
        'low_quality_count': len(assessment.low_quality_docs),
        'next_steps': ' | '.join(assessment.next_steps),
    })

df = pd.DataFrame(rows)
df.sort_values(['status', 'completeness_score']).head(10)

## 7. Retrieval-Like Guidance Templates

In production, next-step guidance is often retrieved from standard operating templates.
We simulate that with a small guidance library keyed by document issue and case state.

In [ ]:
GUIDANCE_LIBRARY = {
    'missing_docs': 'Send document request checklist with direct upload link and deadline.',
    'expired_docs': 'Ask for a renewed valid document and note that onboarding cannot complete with expired ID.',
    'review_docs': 'Route case to manual verification queue with current uploaded artifact attached.',
    'low_quality_docs': 'Send re-upload request with image-quality examples and accepted formats.',
    'ready': 'Trigger activation workflow: provisioning, orientation invite, and policy package.',
    'approaching_start': 'Escalate same-day or next-day starters to operations lead for intervention.',
}

def retrieve_guidance_templates(assessment: CaseAssessment) -> list[str]:
    templates = []
    if assessment.status == 'ready':
        templates.append(GUIDANCE_LIBRARY['ready'])
    if assessment.missing_docs:
        templates.append(GUIDANCE_LIBRARY['missing_docs'])
    if assessment.expired_docs:
        templates.append(GUIDANCE_LIBRARY['expired_docs'])
    if assessment.review_docs:
        templates.append(GUIDANCE_LIBRARY['review_docs'])
    if assessment.low_quality_docs:
        templates.append(GUIDANCE_LIBRARY['low_quality_docs'])
    days_to_start = (datetime.fromisoformat(assessment.case.start_date).date() - BASE_DATE.date()).days
    if days_to_start <= 3 and assessment.status != 'ready':
        templates.append(GUIDANCE_LIBRARY['approaching_start'])
    return templates

for assessment in assessments[:3]:
    print(assessment.case.case_id)
    for template in retrieve_guidance_templates(assessment):
        print('  -', template)

## 8. Personalised Message Generation

We can now transform assessment outputs into applicant-facing or operations-facing messages.
This is where an LLM or templating system usually sits: converting structured findings into concise, actionable guidance.

In [ ]:
def build_applicant_message(assessment: CaseAssessment) -> str:
    greeting = f'Hello {assessment.case.full_name},'
    intro = 'We reviewed your onboarding submission and identified the following next steps:'
    bullets = []
    if assessment.missing_docs:
        bullets.append('Please upload: ' + ', '.join(human_name(d) for d in assessment.missing_docs) + '.')
    if assessment.expired_docs:
        bullets.append('Please replace expired documents: ' + ', '.join(human_name(d) for d in assessment.expired_docs) + '.')
    if assessment.low_quality_docs:
        bullets.append('Please re-upload clearer files for: ' + ', '.join(human_name(d) for d in assessment.low_quality_docs) + '.')
    if assessment.review_docs and not assessment.missing_docs and not assessment.expired_docs:
        bullets.append('Your submitted documents are under review. No additional action is needed unless we contact you.')
    if assessment.status == 'ready':
        bullets.append('Your onboarding packet is complete. We will send activation details shortly.')
    closing = 'Thank you for helping us complete your onboarding promptly.'
    return '
'.join([greeting, '', intro] + [f'- {item}' for item in bullets] + ['', closing])

def build_ops_message(assessment: CaseAssessment) -> str:
    return (
        f'{assessment.case.case_id} | {assessment.case.full_name} | status={assessment.status} | '
        f'score={assessment.completeness_score} | next={'; '.join(assessment.next_steps)}'
    )

case_example = next(a for a in assessments if a.status == 'blocked')
print(build_applicant_message(case_example))
print('
--- OPS VIEW ---
')
print(build_ops_message(case_example))

## 9. Operational Prioritisation

Operations teams need a queue ordered by urgency.
We prioritise by combining completeness, start-date proximity, and workflow risk.

In [ ]:
def priority_score(assessment: CaseAssessment) -> float:
    days_to_start = max(0, (datetime.fromisoformat(assessment.case.start_date).date() - BASE_DATE.date()).days)
    urgency = 1 / (days_to_start + 1)
    risk_weight = {'high': 1.0, 'medium': 0.6, 'low': 0.2}[assessment.risk_level]
    status_weight = {'blocked': 1.0, 'review_needed': 0.65, 'ready': 0.1}[assessment.status]
    return round(0.45 * (1 - assessment.completeness_score) + 0.35 * urgency + 0.20 * max(risk_weight, status_weight), 3)

df['priority_score'] = [priority_score(a) for a in assessments]
priority_queue = df.sort_values('priority_score', ascending=False)
priority_queue[['case_id', 'name', 'status', 'risk_level', 'start_date', 'priority_score']].head(10)

## 10. Evaluation

We evaluate whether the workflow classifies cases into usable operational buckets.
For demonstration, we define an expert-labeled expected status for a subset of cases.

In [ ]:
EXPECTED_STATUS = {
    'ONB-1001': 'ready',
    'ONB-1002': 'blocked',
    'ONB-1003': 'blocked',
    'ONB-1004': 'blocked',
    'ONB-1005': 'ready',
    'ONB-1006': 'review_needed',
    'ONB-1007': 'blocked',
    'ONB-1008': 'review_needed',
    'ONB-1009': 'ready',
    'ONB-1010': 'blocked',
}

matches = []
for assessment in assessments:
    expected = EXPECTED_STATUS.get(assessment.case.case_id)
    if expected is None:
        continue
    matches.append({
        'case_id': assessment.case.case_id,
        'expected': expected,
        'predicted': assessment.status,
        'correct': expected == assessment.status,
    })

eval_df = pd.DataFrame(matches)
accuracy = eval_df['correct'].mean()
print(f'Status accuracy: {accuracy:.1%}')
eval_df

## 11. Visualisations

The following charts make the onboarding pipeline operationally interpretable.

In [ ]:
fig1 = px.histogram(
    df, x='status', color='status',
    category_orders={'status': ['blocked', 'review_needed', 'ready']},
    color_discrete_map={'blocked': '#d62728', 'review_needed': '#ff7f0e', 'ready': '#2ca02c'},
    title='Case Status Distribution'
)
fig1.update_layout(showlegend=False, height=380)
fig1.show()

In [ ]:
status_region = df.groupby(['region', 'status']).size().reset_index(name='count')
fig2 = px.bar(
    status_region, x='region', y='count', color='status', barmode='group',
    color_discrete_map={'blocked': '#d62728', 'review_needed': '#ff7f0e', 'ready': '#2ca02c'},
    title='Status by Region'
)
fig2.update_layout(height=400)
fig2.show()

In [ ]:
issue_totals = pd.DataFrame({
    'issue_type': ['missing', 'review', 'expired', 'low_quality'],
    'count': [df['missing_count'].sum(), df['review_count'].sum(), df['expired_count'].sum(), df['low_quality_count'].sum()],
})
fig3 = px.bar(issue_totals, x='issue_type', y='count', color='issue_type', title='Document Issue Totals')
fig3.update_layout(showlegend=False, height=380)
fig3.show()

In [ ]:
fig4 = px.scatter(
    df, x='completeness_score', y='priority_score', color='status', size='missing_count',
    hover_data=['case_id', 'name', 'start_date'],
    color_discrete_map={'blocked': '#d62728', 'review_needed': '#ff7f0e', 'ready': '#2ca02c'},
    title='Completeness vs Priority Score'
)
fig4.update_layout(height=400)
fig4.show()

In [ ]:
role_heatmap = df.pivot_table(index='role_family', columns='status', values='case_id', aggfunc='count', fill_value=0)
fig5 = go.Figure(data=go.Heatmap(
    z=role_heatmap.values, x=list(role_heatmap.columns), y=list(role_heatmap.index),
    colorscale='Blues', text=role_heatmap.values, texttemplate='%{text}'
))
fig5.update_layout(title='Role Family by Case Status', height=420)
fig5.show()

In [ ]:
template_usage = []
for assessment in assessments:
    for template in retrieve_guidance_templates(assessment):
        template_usage.append(template.split('.')[0])
template_counts = pd.DataFrame(Counter(template_usage).items(), columns=['template', 'count']).sort_values('count', ascending=False)
fig6 = px.bar(template_counts, x='count', y='template', orientation='h', title='Guidance Template Usage')
fig6.update_layout(height=450)
fig6.show()

In [ ]:
eval_summary = pd.DataFrame({
    'metric': ['status_accuracy', 'avg_completeness', 'blocked_share', 'ready_share'],
    'value': [
        accuracy * 100,
        df['completeness_score'].mean() * 100,
        (df['status'] == 'blocked').mean() * 100,
        (df['status'] == 'ready').mean() * 100,
    ]
})
fig7 = px.bar(eval_summary, x='metric', y='value', color='metric', title='Workflow Metrics Summary')
fig7.update_layout(showlegend=False, height=380, yaxis_title='Percent')
fig7.show()

## 12. Production Design

A production onboarding workflow would typically include:

| Layer | Typical implementation |
|---|---|
| Intake collection | Web form, CRM, ATS, HRIS integration |
| Document ingestion | Object storage + OCR/document parser |
| Rules engine | Required-document matrix by region, contract type, and role |
| Verification queue | Manual review UI for ambiguous or failed cases |
| Guidance generation | LLM or template engine using structured case facts |
| Notification layer | Email, SMS, Slack, ticketing |
| Analytics | Throughput, completion rate, follow-up load, SLA tracking |

The core design principle is separation of concerns: rules and status should remain deterministic; generated language should explain decisions, not invent them.

### Reference Production Pattern

Below is an example of how this workflow can be implemented with a rules layer plus an LLM explanation layer.

In [ ]:
PRODUCTION_SNIPPET = '''
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

class GuidanceResponse(BaseModel):
    applicant_message: str
    ops_summary: str
    next_steps: list[str]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """You are generating onboarding guidance from structured case facts.

    Facts:
    - case_id: {case_id}
    - onboarding_type: {onboarding_type}
    - status: {status}
    - completeness_score: {score}
    - missing_docs: {missing_docs}
    - review_docs: {review_docs}
    - expired_docs: {expired_docs}
    - low_quality_docs: {low_quality_docs}

    Requirements:
    1. Do not invent documents not present in the facts.
    2. Keep applicant guidance concise and actionable.
    3. Keep ops summary operational, not conversational.
    4. Preserve the workflow status decided by the rules engine.
    """
)

structured_llm = llm.with_structured_output(GuidanceResponse)
result = (prompt | structured_llm).invoke({...})
'''

print(PRODUCTION_SNIPPET)

## Summary

This notebook built a complete onboarding workflow that:

- collects onboarding details and document submissions
- determines required-document bundles by onboarding type, role, and region
- checks completeness, review state, expiry, and upload quality
- classifies cases into `ready`, `review_needed`, and `blocked`
- generates specific next-step guidance for applicants and operations teams
- prioritises follow-up work and visualises operational bottlenecks

The important architectural split is simple: deterministic workflow logic decides status; generation explains the result and proposes the next move.